# INITIAL INSPECTION

In [2]:
import pandas as pd

root = '../../../'
input_path = root + "raw-data/migration/emigrations_frederiksberg.csv"

df = pd.read_csv(
    input_path,
    sep=";",
    encoding="latin1",
    engine="python",
    skiprows=1
)

# Clean column names
df.columns = df.columns.str.strip()
cols = df.columns.tolist()
cols[0] = "region"
cols[1] = "sex"
cols[2] = "age_group"
cols[3] = "citizenship"
df.columns = cols

# Keep only selected years
df.columns = df.columns.astype(str).str.strip()
years_to_keep = ["2009", "2013", "2017", "2021"]
df_small = df[["region", "sex", "age_group", "citizenship"] + years_to_keep]

df_small.head(20)


,region,sex,age_group,citizenship,2009,2013,2017,2021
0,Frederiksberg,None,None,None,NaN,NaN,NaN,NaN
1,,Men,None,None,NaN,NaN,NaN,NaN
2,,,0-9 years,None,NaN,NaN,NaN,NaN
3,,,,Denmark,40.0,74.0,40.0,30.0
4,,,,Albania,0.0,0.0,0.0,0.0
5,,,,Andorra,0.0,0.0,0.0,0.0
6,,,,Belarus,0.0,0.0,0.0,0.0
7,,,,Belgium,0.0,0.0,0.0,1.0
8,,,,Bosnia and Herzegovina,0.0,0.0,0.0,0.0
9,,,,Bulgaria,0.0,1.0,0.0,0.0


# DROPPING AGES THAT CAN'T VOTE

In [3]:
df_small["age_group"].unique()


array([None, '0-9 years', ' ', '10-19 years', '20-29 years',
       '30-39 years', '40-49 years', '50-59 years', '60-69 years',
       '70-79 years', '80-89 years', '90-99 years'], dtype=object)

In [4]:
df2 = df_small.copy()

# Treat single-space strings as missing
for col in ["region", "sex", "age_group", "citizenship"]:
    df2[col] = df2[col].replace(" ", pd.NA)

# 1) Forward-fill region, sex, and age_group so every row inherits the right values
df2[["region", "sex", "age_group"]] = df2[["region", "sex", "age_group"]].ffill()

# 2) Keep only rows with a real citizenship (these rows have the numbers)
df2 = df2[
    df2["citizenship"].notna() &
    (df2["citizenship"].str.strip() != "")
]

# 3) Drop age groups 0–9 and 10–19 (both hyphen and en-dash variants)
ages_to_drop = ["0–9 years", "10–19 years", "0-9 years", "10-19 years"]
df2 = df2[~df2["age_group"].isin(ages_to_drop)]

# Optional: reset index for neatness
df2 = df2.reset_index(drop=True)

df2.head(20)



,region,sex,age_group,citizenship,2009,2013,2017,2021
0,Frederiksberg,Men,20-29 years,Denmark,141.0,126.0,134.0,113.0
1,Frederiksberg,Men,20-29 years,Albania,1.0,3.0,1.0,1.0
2,Frederiksberg,Men,20-29 years,Andorra,0.0,0.0,0.0,0.0
3,Frederiksberg,Men,20-29 years,Belarus,0.0,0.0,1.0,0.0
4,Frederiksberg,Men,20-29 years,Belgium,5.0,5.0,5.0,6.0
5,Frederiksberg,Men,20-29 years,Bosnia and Herzegovina,0.0,1.0,0.0,0.0
6,Frederiksberg,Men,20-29 years,Bulgaria,1.0,8.0,3.0,3.0
7,Frederiksberg,Men,20-29 years,Cyprus,0.0,0.0,0.0,1.0
8,Frederiksberg,Men,20-29 years,GDR,0.0,0.0,0.0,0.0
9,Frederiksberg,Men,20-29 years,Estonia,1.0,1.0,1.0,0.0


# AGGREGATING GENDERS

In [5]:
df2["sex"].unique()

array(['Men', 'Women'], dtype=object)

In [6]:
years = ["2009", "2013", "2017", "2021"]

df_no_sex = (
    df2
    .groupby(["region", "age_group", "citizenship"], as_index=False)[years]
    .sum()
)

df_no_sex.head(20)


,region,age_group,citizenship,2009,2013,2017,2021
0,Frederiksberg,20-29 years,Abu Dhabi,0.0,0.0,0.0,0.0
1,Frederiksberg,20-29 years,Afghanistan,0.0,0.0,0.0,1.0
2,Frederiksberg,20-29 years,Africa not stated,0.0,0.0,0.0,0.0
3,Frederiksberg,20-29 years,Albania,1.0,4.0,1.0,1.0
4,Frederiksberg,20-29 years,Algeria,0.0,0.0,0.0,1.0
5,Frederiksberg,20-29 years,Andorra,0.0,0.0,0.0,0.0
6,Frederiksberg,20-29 years,Angola,0.0,0.0,0.0,0.0
7,Frederiksberg,20-29 years,Antigua and Barbuda,0.0,0.0,0.0,0.0
8,Frederiksberg,20-29 years,Argentina,1.0,1.0,10.0,47.0
9,Frederiksberg,20-29 years,Armenia,1.0,0.0,1.0,0.0


# SANITY CHECKS FOR COMBINING DATA

In [7]:
years = ["2009", "2013", "2017", "2021"]

# Original rows with sex
check = df2[
    (df2["region"] == "Frederiksberg") &
    (df2["age_group"] == "20-29 years") &
    (df2["citizenship"] == "Austria")
][["sex"] + years]
print(check)

# Row in df_no_sex (Men+Women total)
check_total = df_no_sex[
    (df_no_sex["region"] == "Frederiksberg") &
    (df_no_sex["age_group"] == "20-29 years") &
    (df_no_sex["citizenship"] == "Austria")
][years]
print(check_total)


        sex  2009  2013  2017  2021
53      Men   7.0   8.0   9.0   9.0
1981  Women  14.0  22.0  20.0   7.0
    2009  2013  2017  2021
13  21.0  30.0  29.0  16.0


In [8]:
years = ["2009", "2013", "2017", "2021"]

total_with_sex = df2[years].sum()
total_no_sex   = df_no_sex[years].sum()

print("With sex:\n", total_with_sex)
print("Without sex (grouped):\n", total_no_sex)
print("Difference:\n", total_with_sex - total_no_sex)


With sex:
 2009    2357.0
2013    2978.0
2017    3057.0
2021    2030.0
dtype: float64
Without sex (grouped):
 2009    2357.0
2013    2978.0
2017    3057.0
2021    2030.0
dtype: float64
Difference:
 2009    0.0
2013    0.0
2017    0.0
2021    0.0
dtype: float64


In [9]:
years = ["2009", "2013", "2017", "2021"]

from_original = (
    df2.groupby(["region", "age_group", "citizenship"], as_index=False)[years]
       .sum()
)

# Sort both to compare
a = from_original.sort_values(["region", "age_group", "citizenship"]).reset_index(drop=True)
b = df_no_sex.sort_values(["region", "age_group", "citizenship"]).reset_index(drop=True)

print(a.equals(b))   # should print True


True


# AGGREGATING AGES

In [10]:
years = ["2009", "2013", "2017", "2021"]

df_all_ages = (
    df_no_sex
    .groupby(["region", "citizenship"], as_index=False)[years]
    .sum()
)

df_all_ages.head(20)


,region,citizenship,2009,2013,2017,2021
0,Frederiksberg,Abu Dhabi,0.0,0.0,0.0,0.0
1,Frederiksberg,Afghanistan,0.0,1.0,0.0,1.0
2,Frederiksberg,Africa not stated,0.0,0.0,0.0,0.0
3,Frederiksberg,Albania,1.0,4.0,1.0,1.0
4,Frederiksberg,Algeria,0.0,2.0,2.0,1.0
5,Frederiksberg,Andorra,0.0,0.0,0.0,0.0
6,Frederiksberg,Angola,0.0,2.0,0.0,0.0
7,Frederiksberg,Antigua and Barbuda,0.0,0.0,0.0,0.0
8,Frederiksberg,Argentina,2.0,3.0,15.0,66.0
9,Frederiksberg,Armenia,1.0,0.0,3.0,0.0


# SANITY CHECK FOR COMBINING AGES

In [11]:
years = ["2009", "2013", "2017", "2021"]

check = df_no_sex[
    (df_no_sex["region"] == "Frederiksberg") &
    (df_no_sex["citizenship"] == "Austria")
][years]

print(check)

manual_sum = check.sum()
print("Manual total:", manual_sum)

combined = df_all_ages[
    (df_all_ages["region"] == "Frederiksberg") &
    (df_all_ages["citizenship"] == "Austria")
][years]

print("df_all_ages total:", combined)




      2009  2013  2017  2021
13    21.0  30.0  29.0  16.0
254    1.0   6.0   3.0   2.0
495    1.0   1.0   0.0   2.0
736    0.0   1.0   0.0   0.0
977    0.0   0.0   0.0   0.0
1218   0.0   0.0   0.0   0.0
1459   0.0   0.0   0.0   0.0
1700   0.0   0.0   0.0   0.0
Manual total: 2009    23.0
2013    38.0
2017    32.0
2021    20.0
dtype: float64
df_all_ages total:     2009  2013  2017  2021
13  23.0  38.0  32.0  20.0


# COMBINING COUNTRIES INTO ESTABLISHED GROUPS

In [12]:
country_to_region_da = {
    # Africa
    "Ghana": "Africa", "Algeria":"Africa","Angola":"Africa","Benin":"Africa","Botswana":"Africa",
    "Burkina Faso":"Africa","Burundi":"Africa","Cape Verde":"Africa","Cameroon":"Africa",
    "Central African Republic":"Africa","Chad":"Africa","Comoros":"Africa",
    "Congo, Republic":"Africa","Ivory Coast":"Africa","Djibouti":"Africa",
    "Congo, Democratic Republic":"Africa","Egypt":"Africa","Equatorial Guinea":"Africa",
    "Eritrea":"Africa","Eswantini":"Africa","Ethiopia":"Africa","Gabon":"Africa",
    "Gambia":"Africa","Gambia, The":"Africa","Guinea":"Africa","Guinea-Bissau":"Africa",
    "Kenya":"Africa","Lesotho":"Africa","Liberia":"Africa","Libya":"Africa",
    "Madagascar":"Africa","Malawi":"Africa","Mali":"Africa","Mauritania":"Africa",
    "Mauritius":"Africa","Morocco":"Africa","Mozambique":"Africa","Namibia":"Africa",
    "Niger":"Africa","Nigeria":"Africa","Rwanda":"Africa","Sao Tome and Principe":"Africa",
    "Senegal":"Africa","Seychelles":"Africa","Sierra Leone":"Africa","Somalia":"Africa",
    "South Africa":"Africa","South Sudan":"Africa","Sudan":"Africa","Tanzania":"Africa",
    "Togo":"Africa","Tunisia":"Africa","Uganda":"Africa","Zambia":"Africa","Zimbabwe":"Africa",
    "Spanish territories in Africa":"Africa","Southwest Africa":"Africa",
    "Africa not stated":"Africa","Reunion":"Africa","Saint Helena":"Africa",

    # Asia and Oceania
    "Afghanistan":"Asia and Oceania","Australia":"Asia and Oceania","Bahrain":"Asia and Oceania",
    "Bangladesh":"Asia and Oceania","Bhutan":"Asia and Oceania","Brunei":"Asia and Oceania",
    "Cambodia":"Asia and Oceania","China":"Asia and Oceania","Fiji":"Asia and Oceania",
    "India":"Asia and Oceania","Indonesia":"Asia and Oceania","Iran":"Asia and Oceania",
    "Iraq":"Asia and Oceania","Israel":"Asia and Oceania","Japan":"Asia and Oceania",
    "Jordan":"Asia and Oceania","Kazakhstan":"Asia and Oceania","Kiribati":"Asia and Oceania",
    "Kuwait":"Asia and Oceania","Kyrgyzstan":"Asia and Oceania","Laos":"Asia and Oceania",
    "Lebanon":"Asia and Oceania","Malaysia":"Asia and Oceania","Maldives":"Asia and Oceania",
    "Mongolia":"Asia and Oceania","Myanmar":"Asia and Oceania","Nauru":"Asia and Oceania",
    "Nepal":"Asia and Oceania","New Zealand":"Asia and Oceania","North Korea":"Asia and Oceania",
    "Oman":"Asia and Oceania","Pakistan":"Asia and Oceania","Papua New Guinea":"Asia and Oceania",
    "Philippines":"Asia and Oceania","Qatar":"Asia and Oceania","Samoa":"Asia and Oceania",
    "Saudi Arabia":"Asia and Oceania","Singapore":"Asia and Oceania","South Korea":"Asia and Oceania",
    "Sri Lanka":"Asia and Oceania","West bank and Gaza":"Asia and Oceania","Syria":"Asia and Oceania",
    "Tajikistan":"Asia and Oceania","Thailand":"Asia and Oceania",
    "Marshall Islands":"Asia and Oceania","Solomon Islands":"Asia and Oceania",
    "East Timor":"Asia and Oceania","Tonga":"Asia and Oceania","Turkmenistan":"Asia and Oceania",
    "Tuvalu":"Asia and Oceania","United Arab Emirates":"Asia and Oceania","Uzbekistan":"Asia and Oceania",
    "Vanuatu":"Asia and Oceania","Vietnam":"Asia and Oceania","Yemen":"Asia and Oceania",
    "West Bank":"Asia and Oceania","Middle East not stated":"Asia and Oceania",
    "Palestine":"Asia and Oceania","East Jerusalem":"Asia and Oceania","Gaza":"Asia and Oceania",
    "Hong Kong":"Asia and Oceania","Asia not stated":"Asia and Oceania",
    "Trucial Oman":"Asia and Oceania","Taiwan":"Asia and Oceania","Sikkim":"Asia and Oceania",
    "Macao":"Asia and Oceania","Dubai":"Asia and Oceania","Abu Dhabi":"Asia and Oceania",
    "Cook Islands":"Asia and Oceania","Pacific Islands":"Asia and Oceania",
    "French territories in the Pacific":"Asia and Oceania","Indochina":"Asia and Oceania",

    # EU Countries
    "Austria":"EU Countries","Belgium":"EU Countries","Bulgaria":"EU Countries","Cyprus":"EU Countries",
    "Czech Republic":"EU Countries","Estonia":"EU Countries","France":"EU Countries","Germany":"EU Countries",
    "Greece":"EU Countries","Hungary":"EU Countries","Ireland":"EU Countries","Italy":"EU Countries",
    "Latvia":"EU Countries","Lithuania":"EU Countries","Luxembourg":"EU Countries","Malta":"EU Countries",
    "Netherlands":"EU Countries","Poland":"EU Countries","Portugal":"EU Countries","Romania":"EU Countries",
    "Slovakia":"EU Countries","Spain":"EU Countries",

    # North America
    "Bermuda":"North America","Canada":"North America","USA":"North America",
    "North America not stated":"North America","Faroe Islands":"North America",
    "Finland":"North America","Greenland":"North America","Iceland":"North America",
    "Norway":"North America","Sweden":"North America",

    # South and Central America
    "Antigua and Barbuda":"South and Central America","Argentina":"South and Central America",
    "Aruba":"South and Central America","Bahamas":"South and Central America",
    "Barbados":"South and Central America","Belize":"South and Central America",
    "Bolivia":"South and Central America","Brazil":"South and Central America",
    "Chile":"South and Central America","Colombia":"South and Central America",
    "Costa Rica":"South and Central America","Cuba":"South and Central America",
    "Curacao":"South and Central America","Dominica":"South and Central America",
    "Dominican Republic":"South and Central America","Ecuador":"South and Central America",
    "El Salvador":"South and Central America","French Guiana":"South and Central America",
    "Grenada":"South and Central America","Guadeloupe":"South and Central America",
    "Guatemala":"South and Central America","Guyana":"South and Central America",
    "Haiti":"South and Central America","Honduras":"South and Central America",
    "Jamaica":"South and Central America","Martinique":"South and Central America",
    "Mexico":"South and Central America","Nicaragua":"South and Central America",
    "Panama":"South and Central America","Paraguay":"South and Central America",
    "Peru":"South and Central America","Puerto Rico":"South and Central America",
    "Saint Kitts and Nevis":"South and Central America","Saint Lucia":"South and Central America",
    "Saint Vincent and the Grenadines":"South and Central America","Suriname":"South and Central America",
    "Falkland Islands":"South and Central America","Trinidad and Tobago":"South and Central America",
    "Uruguay":"South and Central America","Venezuela":"South and Central America",
    "South and central America not stated":"South and Central America",
    "British West Indies":"South and Central America","West Indies":"South and Central America",
    "French West Indies":"South and Central America","Netherlands Antilles":"South and Central America",

    # Former Yugoslavia
    "Bosnia and Herzegovina":"Former Yugoslavia","Kosovo":"Former Yugoslavia",
    "Croatia":"Former Yugoslavia","Montenegro":"Former Yugoslavia",
    "Republic of North Macedonia":"Former Yugoslavia","Serbia":"Former Yugoslavia",
    "Slovenia":"Former Yugoslavia","Yugoslavia":"Former Yugoslavia",
    "Yugoslavia, Federal Republic":"Former Yugoslavia",
    "Serbia and Montenegro":"Former Yugoslavia",

    # Turkey
    "Turkey":"Turkey",

    # Unknown
    "Stateless":"Unknown","Soviet Union":"Unknown","Czechoslovakia":"Unknown",
    "Not stated":"Unknown","GDR":"Unknown","North Yemen":"Unknown",

    # Other Europe
    "Albania":"Other Europe","Andorra":"Other Europe","Armenia":"Other Europe",
    "Azerbaijan":"Other Europe","Belarus":"Other Europe","Georgia":"Other Europe",
    "Liechtenstein":"Other Europe","Moldova":"Other Europe","Monaco":"Other Europe",
    "Russia":"Other Europe","San Marino":"Other Europe","Switzerland":"Other Europe",
    "Ukraine":"Other Europe","United Kingdom":"Other Europe",
    "Vatican City State":"Other Europe","Northern Ireland":"Other Europe",
    "Europe not stated":"Other Europe",

    #Denmark
    "Denmark": "Denmark",

}

df_all_ages["region_group_da"] = df_all_ages["citizenship"].map(country_to_region_da)

years = ["2009", "2013", "2017", "2021"]

df_region_da = (
    df_all_ages
    .groupby(["region", "region_group_da"], as_index=False)[years]
    .sum()
)

df_region_da


,region,region_group_da,2009,2013,2017,2021
0,Frederiksberg,Africa,44.0,54.0,46.0,24.0
1,Frederiksberg,Asia and Oceania,326.0,477.0,622.0,248.0
2,Frederiksberg,Denmark,558.0,671.0,566.0,445.0
3,Frederiksberg,EU Countries,794.0,789.0,895.0,657.0
4,Frederiksberg,Former Yugoslavia,12.0,12.0,10.0,18.0
5,Frederiksberg,North America,451.0,745.0,657.0,435.0
6,Frederiksberg,Other Europe,135.0,141.0,154.0,100.0
7,Frederiksberg,South and Central America,27.0,65.0,84.0,95.0
8,Frederiksberg,Turkey,9.0,22.0,19.0,7.0
9,Frederiksberg,Unknown,1.0,2.0,4.0,1.0


# CHECK IF ALL COUNTRIES GOT ASSIGNED

In [13]:
missing = df_all_ages[df_all_ages["region_group_da"].isna()]
missing


,region,citizenship,2009,2013,2017,2021,region_group_da


# CHANGE LAYOUT

In [14]:
df_regions = df_region_da.drop(columns=["region"])

years = ["2009", "2013", "2017", "2021"]

# 1. Drop the region column
df_regions = df_region_da.drop(columns=["region"])

# 2. Melt to long format
df_long = df_regions.melt(
    id_vars="region_group_da",
    value_vars=years,
    var_name="year",
    value_name="value"
)

# 3. Pivot so regions become columns
df_pivot = df_long.pivot(
    index="year",
    columns="region_group_da",
    values="value"
)

# 4. Optional: sort columns alphabetically
df_pivot = df_pivot.sort_index(axis=1)

df_pivot


region_group_da,Africa,Asia and Oceania,Denmark,EU Countries,Former Yugoslavia,North America,Other Europe,South and Central America,Turkey,Unknown
year,,,,,,,,,,
2009,44.0,326.0,558.0,794.0,12.0,451.0,135.0,27.0,9.0,1.0
2013,54.0,477.0,671.0,789.0,12.0,745.0,141.0,65.0,22.0,2.0
2017,46.0,622.0,566.0,895.0,10.0,657.0,154.0,84.0,19.0,4.0
2021,24.0,248.0,445.0,657.0,18.0,435.0,100.0,95.0,7.0,1.0


In [15]:
relative_df = df_pivot.div(df_pivot.sum(axis=1), axis=0) * 100
relative_df = relative_df.round(1)
relative_df.head()

region_group_da,Africa,Asia and Oceania,Denmark,EU Countries,Former Yugoslavia,North America,Other Europe,South and Central America,Turkey,Unknown
year,,,,,,,,,,
2009,1.9,13.8,23.7,33.7,0.5,19.1,5.7,1.1,0.4,0.0
2013,1.8,16.0,22.5,26.5,0.4,25.0,4.7,2.2,0.7,0.1
2017,1.5,20.3,18.5,29.3,0.3,21.5,5.0,2.7,0.6,0.1
2021,1.2,12.2,21.9,32.4,0.9,21.4,4.9,4.7,0.3,0.0


# EXPORT CSV

In [16]:
output_path = root + "processed-data/migration/frb_emigration_by_region.csv"
df_pivot.to_csv(output_path)

In [17]:
output_path = root + "processed-data/migration/relative_frb_emigration_by_region.csv"
relative_df.to_csv(output_path)